# Theorem 8 — shared estimation perturbation bound

**Formal source:** [`../08_shared_estimation_perturbation_bound.md`](../08_shared_estimation_perturbation_bound.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
operator = np.array([[1, 0], [0, 1], [1, 1], [2, -1]], float)
covariance = np.diag([0.1, 1, 4, 0.2])
state = np.array([1.5, -0.7])
noise = np.array([0.02, -0.15, 0.4, -0.03])
observation = operator @ state + noise
cholesky = np.linalg.cholesky(covariance)
whitened = np.linalg.solve(cholesky, operator)
whitened_noise = np.linalg.solve(cholesky, noise)
gamma = np.linalg.svd(whitened, compute_uv=False)[-1]
weight = np.linalg.inv(covariance)
estimate = np.linalg.solve(operator.T @ weight @ operator, operator.T @ weight @ observation)
error = np.linalg.norm(estimate - state)
bound = np.linalg.norm(whitened_noise) / gamma
assert error <= bound + 1e-12
print({"estimate": estimate.tolist(), "error": float(error), "bound": float(bound), "gamma": float(gamma)})

In [ ]:
print('THEORY_DEMO_PASS::08_shared_estimation_perturbation_bound')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')